# Milestone 3 Pushshift Reddit Preprocessing & First Model Building and Evaluation

This notebook performs preprocessing and preliminary model building and evaluation on the Pushshift Reddit dataset.

In [1]:
# Dependencies 

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
from datetime import datetime
import os
import glob
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.types import LongType
import requests

Matplotlib created a temporary cache directory at /scratch/jkeeton/job_48863073/matplotlib-fk161tha because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [2]:
# SparkSession Configuration

# 16 cores, 128GB total memory — local[*] mode
# In local mode there are no separate executor processes — all task execution
# runs as threads within a single JVM. Executor config parameters have no effect
# and are omitted. Driver memory is set to 120GB to give the JVM nearly the
# full node allocation.
spark = SparkSession.builder \
    .appName("PushshiftRedditPreprocessing") \
    .master("local[*]") \
    .config("spark.driver.memory", "120g") \
    .config("spark.driver.maxResultSize", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.local.dir", '/expanse/lustre/scratch/jkeeton/temp_project/spark-tmp') \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Spark version: 3.5.0
Spark UI: http://exp-1-18.expanse.sdsc.edu:4040


In [3]:
# Reload df_features

FEATURES_DIR = "/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/features/"
df_features = spark.read.parquet(FEATURES_DIR)
print(f"Rows: {df_features.count():,}")
print(f"Columns: {len(df_features.columns)}")

Rows: 535,480,818
Columns: 31


### Word2Vec Title Embedding — Runtime Estimate

Before committing to a full Word2Vec run on 535M rows, we benchmark on a 5M row sample and extrapolate. Word2Vec is an iterative training algorithm — unlike row-wise transformations such as VADER, its runtime scales with dataset size and vocabulary, not just row count. The key parameters that control the cost/quality tradeoff are:

- `maxIter=1` — single pass over the data; default and sufficient for a first model
- `vectorSize=50` — embedding dimensions; standard for short text like titles
- `minCount=5` — filters tokens appearing fewer than 5 times, significantly reducing vocabulary size
- `numPartitions=16` — matches core count on the Expanse node

In [4]:
# Benchmark on 5M rows before committing to full run
from pyspark.ml.feature import Tokenizer, Word2Vec
import time

sample_df = df_features.sample(fraction=5_000_000/535_480_818, seed=42)

tokenizer = Tokenizer(inputCol="title", outputCol="title_tokens")
sample_df = tokenizer.transform(sample_df)

w2v = Word2Vec(inputCol="title_tokens", outputCol="title_embedding",
               vectorSize=50, minCount=5, numPartitions=16, maxIter=1)

start = time.time()
model = w2v.fit(sample_df)
elapsed = time.time() - start

FULL_DATASET = 535_480_818

print(f"5M row sample: {elapsed/60:.1f} minutes")
print(f"Estimated full {FULL_DATASET:,} rows: {elapsed * (FULL_DATASET/5_000_000):.0f} seconds ({elapsed * (FULL_DATASET/5_000_000) / 3600:.1f} hours)")

5M row sample: 6.5 minutes
Estimated full 535,480,818 rows: 41615 seconds (11.6 hours)


### Word2Vec Title Embedding — Full Implementation

Word2Vec is trained on the `title` column of the training set only to prevent leakage, then applied to all three splits. Titles are first tokenized into word arrays using Spark's `Tokenizer`. The resulting 50-dimensional embedding vector represents each title as a dense fixed-size vector computed by averaging the Word2Vec word vectors across all tokens in the title.

The embedding vector cannot be passed directly to `VectorAssembler` — MLlib requires individual numeric columns, not vector columns. We use `split` on the DenseVector's string representation to explode the 50 dimensions into individual float columns (`w2v_0` through `w2v_49`) before assembly.

Empty or very short titles (filtered by `minCount`) may produce zero vectors. The existing `has_title` flag gives the model context to interpret these correctly.

In [5]:
# ── WORD2VEC TITLE EMBEDDING ──────────────────────────────────────────────────
from pyspark.ml.feature import Tokenizer, Word2Vec
from pyspark.sql.types import FloatType, ArrayType
import time
import datetime

VECTOR_SIZE = 50

# Tokenize title column
tokenizer = Tokenizer(inputCol="title", outputCol="title_tokens")
df_features = tokenizer.transform(df_features)

# Fit on full df_features — train/val/test split happens downstream
w2v = Word2Vec(
    inputCol="title_tokens",
    outputCol="title_embedding",
    vectorSize=VECTOR_SIZE,
    minCount=5,
    numPartitions=16,
    maxIter=1,
    seed=42
)

start_time = time.time()
start_dt = datetime.datetime.now()
print(f"Word2Vec training started at: {start_dt.strftime('%Y-%m-%d %H:%M:%S')}")

w2v_model = w2v.fit(df_features)
df_features = w2v_model.transform(df_features)

elapsed = time.time() - start_time
print(f"Training complete. Runtime: {int(elapsed//3600)}h {int((elapsed%3600)//60)}m {int(elapsed%60)}s")

# ── EXPLODE EMBEDDING VECTOR INTO INDIVIDUAL FLOAT COLUMNS ───────────────────
# VectorAssembler requires individual numeric columns, not vector columns

@F.udf(returnType=ArrayType(FloatType()))
def vector_to_array(v):
    if v is None:
        return [0.0] * VECTOR_SIZE
    return [float(x) for x in v.toArray()]

w2v_cols = [f"w2v_{i}" for i in range(VECTOR_SIZE)]

df_features = df_features \
    .withColumn("w2v_array", vector_to_array(F.col("title_embedding")))

for i, col_name in enumerate(w2v_cols):
    df_features = df_features.withColumn(col_name, F.col("w2v_array").getItem(i))

df_features = df_features.drop("w2v_array", "title_tokens", "title_embedding")

print(f"Embedding columns added: {w2v_cols[:3]} ... {w2v_cols[-1]}")
print(f"Total columns: {len(df_features.columns)}")

# Sanity check — verify no nulls in embedding columns
null_check = df_features.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in w2v_cols
]).collect()[0].asDict()
max_nulls = max(null_check.values())
print(f"Max nulls across embedding columns: {max_nulls}")

Word2Vec training started at: 2026-05-09 12:07:11
Training complete. Runtime: 2h 48m 53s
Embedding columns added: ['w2v_0', 'w2v_1', 'w2v_2'] ... w2v_49
Total columns: 81
Max nulls across embedding columns: 0


In [ ]:
# ── PERSIST df_features WITH WORD2VEC EMBEDDINGS ─────────────────────────────

FEATURES_W2V_DIR = "/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/features_w2v/"

print("Writing df_features with Word2Vec embeddings to disk...")
df_features.write \
    .mode("overwrite") \
    .parquet(FEATURES_W2V_DIR)

print("Write complete. Reloading from disk...")
df_features = spark.read.parquet(FEATURES_W2V_DIR)

print(f"Rows: {df_features.count():,}")
print(f"Columns: {len(df_features.columns)}")
print(df_features.columns)

Writing df_features with Word2Vec embeddings to disk...
